In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interactive, FloatSlider, RadioButtons, HBox, Layout, VBox, HTML, GridBox
from IPython.display import display

# ============================================================
# INTERACTIVE FUNCTION
# ============================================================

def plot_autocorrelation_psd(correlation_type='Exponential', alpha=1.0, T=2.0, sigma_tau=1.0):

    # --------------------------------------------------------
    # VARIABLES
    # --------------------------------------------------------

    tau = np.linspace(-8.0, 8.0, 2000)

    omega = np.linspace(-10.0, 10.0, 2000)

    # ========================================================
    # EXPONENTIAL AUTOCORRELATION
    #
    # R(tau) = exp(-alpha |tau|)
    #
    # S(omega) = 2 alpha / (alpha^2 + omega^2)
    # ========================================================

    if correlation_type == 'Exponential':

        R = np.exp(-alpha * np.abs(tau))

        S = (
            2.0 * alpha
            /
            (alpha ** 2 + omega ** 2)
        )

        process_name = f'Exponential Autocorrelation, α = {alpha:.2f}'

    # ========================================================
    # TRIANGULAR AUTOCORRELATION
    #
    # R(tau) = max(1 - |tau|/T, 0)
    #
    # S(omega) = T sinc^2(omega T / 2)
    # ========================================================

    elif correlation_type == 'Triangular':

        R = np.maximum(
            1.0 - np.abs(tau) / T,
            0.0
        )

        x = omega * T / 2.0

        S = np.ones_like(x)

        nonzero = np.abs(x) > 1e-12

        S[nonzero] = (
            np.sin(x[nonzero])
            /
            x[nonzero]
        ) ** 2

        S = T * S

        process_name = f'Triangular Autocorrelation, T = {T:.2f}'

    # ========================================================
    # GAUSSIAN AUTOCORRELATION
    #
    # R(tau) = exp[-tau^2/(2 sigma_tau^2)]
    #
    # Fourier transform is also Gaussian
    # ========================================================

    else:

        R = np.exp(
            -(tau ** 2)
            /
            (2.0 * sigma_tau ** 2)
        )

        S = (
            np.sqrt(2.0 * np.pi)
            * sigma_tau
            * np.exp(
                -0.5
                * sigma_tau ** 2
                * omega ** 2
            )
        )

        process_name = f'Gaussian Autocorrelation, στ = {sigma_tau:.2f}'

    # --------------------------------------------------------
    # NORMALIZATION
    #
    # We normalize both functions to emphasize the WIDTH
    # relationship rather than their absolute amplitudes.
    # --------------------------------------------------------

    R_normalized = R / np.max(R)

    S_normalized = S / np.max(S)

    # --------------------------------------------------------
    # FIGURE
    # --------------------------------------------------------

    fig, (ax1, ax2) = plt.subplots(
        2,
        1,
        figsize=(8.0, 5.6)
    )

    # ========================================================
    # GRAPH 1:
    # AUTOCORRELATION
    # ========================================================

    ax1.plot(
        tau,
        R_normalized,
        linewidth=2
    )

    ax1.axhline(
        0,
        color='k',
        linewidth=0.8
    )

    ax1.axvline(
        0,
        color='k',
        linestyle=':',
        linewidth=0.8
    )

    ax1.set_xlim(
        -8.0,
        8.0
    )

    ax1.set_ylim(
        -0.05,
        1.10
    )

    ax1.set_xlabel(
        'Time lag τ',
        fontsize=11
    )

    ax1.set_ylabel(
        'Normalized Rₓ(τ)',
        fontsize=11
    )

    ax1.set_title(
        process_name,
        fontsize=12,
        pad=7
    )

    ax1.tick_params(
        axis='both',
        labelsize=9
    )

    ax1.grid(
        True,
        linestyle=':',
        alpha=0.6
    )

    # ========================================================
    # GRAPH 2:
    # POWER SPECTRAL DENSITY
    # ========================================================

    ax2.plot(
        omega,
        S_normalized,
        linewidth=2
    )

    ax2.axhline(
        0,
        color='k',
        linewidth=0.8
    )

    ax2.axvline(
        0,
        color='k',
        linestyle=':',
        linewidth=0.8
    )

    ax2.set_xlim(
        -10.0,
        10.0
    )

    ax2.set_ylim(
        -0.05,
        1.10
    )

    ax2.set_xlabel(
        'Angular frequency ω',
        fontsize=11
    )

    ax2.set_ylabel(
        'Normalized Sₓ(ω)',
        fontsize=11
    )

    ax2.set_title(
        'Corresponding Power Spectral Density',
        fontsize=12,
        pad=7
    )

    ax2.tick_params(
        axis='both',
        labelsize=9
    )

    ax2.grid(
        True,
        linestyle=':',
        alpha=0.6
    )

    # ========================================================
    # FIGURE SPACING
    # ========================================================

    plt.subplots_adjust(
        left=0.12,
        right=0.97,
        top=0.95,
        bottom=0.10,
        hspace=0.42
    )

    plt.show()

# ============================================================
# RADIO BUTTONS
# ============================================================

correlation_selector = RadioButtons(
    options=[
        'Exponential',
        'Triangular',
        'Gaussian'
    ],
    value='Exponential',
    description='Correlation:',
    style={'description_width': 'initial'},
    layout=Layout(width='270px')
)

# ============================================================
# SLIDERS
# ============================================================

slider_style = {
    'description_width': '0px'
}

slider_layout = Layout(
    width='100px'
)

alpha_slider = FloatSlider(
    min=0.2,
    max=3.0,
    step=0.1,
    value=1.0,
    description=' ',
    continuous_update=True,
    readout=False,
    style=slider_style,
    layout=slider_layout
)

T_slider = FloatSlider(
    min=0.5,
    max=5.0,
    step=0.1,
    value=2.0,
    description=' ',
    continuous_update=True,
    readout=False,
    style=slider_style,
    layout=slider_layout,
    disabled=True
)

sigma_slider = FloatSlider(
    min=0.3,
    max=3.0,
    step=0.1,
    value=1.0,
    description=' ',
    continuous_update=True,
    readout=False,
    style=slider_style,
    layout=slider_layout,
    disabled=True
)

# ============================================================
# MAXIMUM VALUE LABELS
# ============================================================

alpha_max_label = HTML(
    '<div style="font-family:Arial; font-size:14px;">3.0</div>'
)

T_max_label = HTML(
    '<div style="font-family:Arial; font-size:14px;">5.0</div>'
)

sigma_max_label = HTML(
    '<div style="font-family:Arial; font-size:14px;">3.0</div>'
)

# ============================================================
# ENABLE / DISABLE SLIDERS
# ============================================================

def update_controls(change):

    alpha_slider.disabled = (
        correlation_selector.value != 'Exponential'
    )

    T_slider.disabled = (
        correlation_selector.value != 'Triangular'
    )

    sigma_slider.disabled = (
        correlation_selector.value != 'Gaussian'
    )

correlation_selector.observe(
    update_controls,
    names='value'
)

# ============================================================
# INTERACTIVE OBJECT
# ============================================================

widget_plot = interactive(
    plot_autocorrelation_psd,
    correlation_type=correlation_selector,
    alpha=alpha_slider,
    T=T_slider,
    sigma_tau=sigma_slider
)

# ============================================================
# DOCUMENTATION
# ============================================================

theory_html = HTML("""
<div style="
    font-family: Arial, sans-serif;
    font-size: 16px;
    line-height: 1.30;
    width: 1050px;
">

<div style="
    font-size: 21px;
    font-weight: bold;
    margin-bottom: 6px;
">
Autocorrelation Shape and Power Spectral Density
</div>

<div style="margin-bottom:5px;">
<b>Exponential:</b> Rₓ(τ) = exp(-α|τ|), whose Fourier transform is a Lorentzian-shaped PSD.
</div>

<div style="margin-bottom:5px;">
<b>Triangular:</b> a finite-width triangular autocorrelation produces a sinc²-shaped PSD.
</div>

<div style="margin-bottom:5px;">
<b>Gaussian:</b> a Gaussian autocorrelation has a Gaussian power spectral density.
</div>

<div style="margin-bottom:5px;">
<b>Wiener–Khinchin:</b> the PSD is obtained as the Fourier transform of the autocorrelation function.
</div>

<div>
<b>This notebook:</b> demonstrates how changing the width of the autocorrelation produces the opposite change in the width of the corresponding PSD.
</div>

</div>
""")

# ============================================================
# EXTRA SPACE BELOW THEORY
# ============================================================

theory_block = VBox(
    [
        theory_html
    ],
    layout=Layout(
        margin='0px 0px 18px 0px'
    )
)

# ============================================================
# LEFT-ALIGNED LABELS
# ============================================================

alpha_label = HTML(
    '<div style="font-family:Arial; font-size:14px;">Decay α:</div>'
)

T_label = HTML(
    '<div style="font-family:Arial; font-size:14px;">Width T:</div>'
)

sigma_label = HTML(
    '<div style="font-family:Arial; font-size:14px;">Width στ:</div>'
)

# ============================================================
# FIXED THREE-COLUMN GRID
# LABEL | SLIDER | MAXIMUM VALUE
# ============================================================

slider_grid = GridBox(
    children=[
        alpha_label, alpha_slider, alpha_max_label,
        T_label, T_slider, T_max_label,
        sigma_label, sigma_slider, sigma_max_label
    ],
    layout=Layout(
        width='265px',
        grid_template_columns='110px 100px 40px',
        grid_template_rows='30px 30px 30px',
        grid_gap='2px 6px',
        align_items='center',
        overflow='hidden'
    )
)

# ============================================================
# SPACE BETWEEN RADIO BUTTONS AND SLIDER GROUP
# ============================================================

slider_group = VBox(
    [
        slider_grid
    ],
    layout=Layout(
        margin='12px 0px 0px 0px'
    )
)

# ============================================================
# CONTROLS
# ============================================================

controls = VBox(
    [
        correlation_selector,
        slider_group
    ],
    layout=Layout(
        width='285px',
        min_width='285px',
        align_items='flex-start',
        justify_content='center',
        margin='0px 0px 0px 10px',
        overflow='hidden'
    )
)

# ============================================================
# FIGURE LEFT - CONTROLS RIGHT
# ============================================================

graph_and_controls = HBox(
    [
        widget_plot.children[-1],
        controls
    ],
    layout=Layout(
        width='1100px',
        align_items='center',
        justify_content='flex-start',
        overflow='hidden'
    )
)

# ============================================================
# INTERPRETATION BELOW FIGURES
# ============================================================

interpretation_html = HTML("""
<div style="
    font-family: Arial, sans-serif;
    font-size: 15px;
    line-height: 1.35;
    width: 1050px;
    margin-top: 12px;
">

<div style="
    font-size: 18px;
    font-weight: bold;
    margin-bottom: 6px;
">
Interpretation of the Results
</div>

<div style="margin-bottom:5px;">
The upper graph shows the autocorrelation function in the time-lag domain, while the lower graph shows its corresponding power spectral density.
</div>

<div style="margin-bottom:5px;">
A <b>rapidly decaying or narrow autocorrelation</b> corresponds to a <b>broad power spectrum</b>.
</div>

<div style="margin-bottom:5px;">
A <b>slowly decaying or wide autocorrelation</b> corresponds to a <b>narrow power spectrum</b>.
</div>

<div style="
    margin-top:8px;
    font-size:17px;
    font-weight:bold;
">
Narrow autocorrelation &nbsp;&nbsp; ⇔ &nbsp;&nbsp; Wide PSD
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
Wide autocorrelation &nbsp;&nbsp; ⇔ &nbsp;&nbsp; Narrow PSD
</div>

</div>
""")

# ============================================================
# COMPLETE LAYOUT
# ============================================================

main_layout = VBox(
    [
        theory_block,
        graph_and_controls,
        interpretation_html
    ],
    layout=Layout(
        width='1100px',
        overflow='hidden'
    )
)

display(main_layout)